<a href="https://colab.research.google.com/github/JohnWendelG/JohnWendelG/blob/main/Copia_de_Proyecto_Listado_Proveedores_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Consolidador de BD Proveedores con CATEGORIA AUTOMATICA

¿QUE HACE EL PROGRAMA?

1.1.-Este programa consolida la imformacion de PROVEEDORES con ciertos campos descargados de la plataforma MONDAYS.

1.2.-Ejecuta una condicion que Actualiza el nivel de Riesgo del proveedor mediante la calificacion o categorizacion del mismo proveedor.

1.3.- Añade proveedores nuevos si asi amerita el caso, ya que al ser un programa automatico detecta todas las variaciones del archivo y las añade como NUEVO.

In [ ]:
# @title
# =========================================================
# CONSOLIDACIÓN DE PROVEEDORES
# - Base + Archivo semanal
# - Match por nombre normalizado (fuerte)
# - Fuzzy matching para nombres parecidos (ALIGNET, etc.)
# - Agrega solo proveedores realmente nuevos del semanal
# =========================================================

!pip -q install openpyxl unidecode rapidfuzz

import re
import pandas as pd
from unidecode import unidecode
from google.colab import files
from rapidfuzz import process, fuzz

# ---------- CONFIGURACIÓN ----------
BASE_CLAVE = ["NOMBRE PROVEEDOR", "PROCEDIMIENTO ASOCIADO", "Nivel de  Criticidad"]
SEMANAL_CLAVE = ["Name", "CATEGORIA"]
# Equivalencias manuales de nombres (después de normalizar)
# Alias definidos "a mano" (forma humana)
ALIAS_MANUALES_RAW = {
    # ALIGNET
    "ALIGNET DEL ECUADOR, ALIGNETSA S.A.": "ALIGNET",
    "ALIGNET S.A.C.": "ALIGNET",

    # PLUS HOLDING
    "Plusholding": "PLUS HOLDING INTERNATIONAL LIMITED",
    "Plusholding (PLUS TECHNOLOGIES AND INNOVATIONS)": "PLUS HOLDING INTERNATIONAL LIMITED",
    "PLUS HOLDING INTERNATIONAL LIMITED": "PLUS HOLDING INTERNATIONAL LIMITED",
}

# Umbral de similitud fuzzy (0–100)
UMBRAL_FUZZY = 80   # puedes subirlo o bajarlo según qué tan agresivo quieres el match

# ---------- FUNCIONES AUXILIARES ----------
def _normalizar_base(nombre: str) -> str:
    """Normalización base sin alias (solo limpieza)."""
    if not isinstance(nombre, str):
        return ""
    nombre = unidecode(str(nombre)).upper().strip()

    # Homologar formas de SAC, CA, SA
    nombre = re.sub(r"\bS\.?\s*A\.?\s*C\.?\b", " SAC", nombre)  # S.A.C., S A C, etc.
    nombre = re.sub(r"\bC\.?\s*A\.?\b", " CA", nombre)          # C.A., C A, etc.
    nombre = re.sub(r"\bS\.?\s*A\.?\b", " SA", nombre)          # S.A., S A, etc.

    # Quitar símbolos raros
    nombre = re.sub(r"[^A-Z0-9\s]", " ", nombre)
    nombre = " ".join(nombre.split())
    return nombre
# Construir alias ya normalizados
ALIAS_MANUALES = {
    _normalizar_base(k): _normalizar_base(v)
    for k, v in ALIAS_MANUALES_RAW.items()
}
def normalizar_nombre(nombre: str) -> str:
    """
    Normaliza nombre y aplica alias manuales.
    """
    nombre_norm = _normalizar_base(nombre)
    # Si hay alias definido, usarlo
    return ALIAS_MANUALES.get(nombre_norm, nombre_norm)



def set_header_by_search(df: pd.DataFrame, columnas_clave, max_rows: int = 5):
    """Busca una fila que contenga las columnas clave y la usa como encabezado."""
    for i in range(min(max_rows, len(df))):
        row = df.iloc[i]
        row_vals = [str(v).strip() for v in row.tolist()]
        if all(col in row_vals for col in columnas_clave):
            header = row_vals
            df2 = df.iloc[i+1:].copy()
            df2.columns = header
            df2 = df2.reset_index(drop=True)
            return df2, i
    return df, None


def cargar_excel_con_dialogo(mensaje: str) -> pd.DataFrame:
    print("\n" + "="*80)
    print(mensaje)
    print("="*80)
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No se subió ningún archivo.")
    filename = list(uploaded.keys())[0]
    print(f"-> Archivo cargado: {filename}")
    df = pd.read_excel(filename)
    return df


def preparar_base(df_raw: pd.DataFrame) -> pd.DataFrame:
    df, idx_header = set_header_by_search(df_raw, BASE_CLAVE, max_rows=5)
    if idx_header is not None:
        print(f"Encabezado de la BASE encontrado en la fila {idx_header}.")
    else:
        print("No se detectó encabezado especial en la BASE; se usan columnas actuales.")
    faltantes = [c for c in BASE_CLAVE if c not in df.columns]
    if faltantes:
        raise ValueError("A la BASE PRINCIPAL le faltan columnas:\n - " + "\n - ".join(faltantes))
    return df


def preparar_semanal(df_raw: pd.DataFrame) -> pd.DataFrame:
    df, idx_header = set_header_by_search(df_raw, SEMANAL_CLAVE, max_rows=5)
    if idx_header is not None:
        print(f"Encabezado del SEMANAL encontrado en la fila {idx_header}.")
    else:
        print("No se detectó encabezado especial en el SEMANAL; se usan columnas actuales.")
    faltantes = [c for c in SEMANAL_CLAVE if c not in df.columns]
    if faltantes:
        raise ValueError("Al ARCHIVO SEMANAL le faltan columnas:\n - " + "\n - ".join(faltantes))
    return df

def calcular_nivel_riesgo(categoria):
    """Mapea la CATEGORIA (A/B/C/D) al nivel de riesgo."""
    if pd.isna(categoria):
        return "REVISAR"
    c = str(categoria).strip().upper()
    if c in ("A", "B"):
        return "Bajo"
    elif c == "C":
        return "Medio"
    elif c == "D":
        return "Alto"
    else:
        return "REVISAR"



# ---------- FLUJO PRINCIPAL ----------

# 1) Cargar archivos
df_base_raw = cargar_excel_con_dialogo(
    "1) Selecciona el archivo BASE PRINCIPAL (Base_Principal_Proveedores.xlsx)"
)
df_base = preparar_base(df_base_raw)
base_cols_original = list(df_base.columns)

df_sem_raw = cargar_excel_con_dialogo(
    "2) Selecciona el archivo SEMANAL (BANCO_GUAYAQUIL_S_A_PROVEEDORES_....xlsx)"
)
df_sem = preparar_semanal(df_sem_raw)

# 2) Llave normalizada
df_base["__NOMBRE_NORM"] = df_base["NOMBRE PROVEEDOR"].apply(normalizar_nombre)
df_sem["__NOMBRE_NORM"] = df_sem["Name"].apply(normalizar_nombre)

# 3) MATCH EXACTO: LEFT JOIN (solo base, se rellena con semanal)
df_join = df_base.merge(
    df_sem,
    on="__NOMBRE_NORM",
    how="left",
    suffixes=("", "_BG")
)

# Asegurar nombre de proveedor principal
if "Name" in df_join.columns:
    df_join["NOMBRE PROVEEDOR"] = df_join["NOMBRE PROVEEDOR"].combine_first(df_join["Name"])

# 4) Determinar qué proveedores del semanal ya se usaron por match exacto
matched_norms_exact = set(df_join["__NOMBRE_NORM"][df_join["Name"].notna()].dropna())
df_sem_unmatched = df_sem[~df_sem["__NOMBRE_NORM"].isin(matched_norms_exact)].copy()

# 5) FUZZY MATCH: base sin match exacto vs semanal no usado
sem_unmatched_dict = df_sem_unmatched.set_index("__NOMBRE_NORM").to_dict("index")
nombres_semanal = list(sem_unmatched_dict.keys())

sin_match = df_join[df_join["Name"].isna() & df_join["__NOMBRE_NORM"].notna()].copy()

matches = []
for idx, row in sin_match.iterrows():
    nb_norm = row["__NOMBRE_NORM"]
    if not nb_norm:
        continue

    if not nombres_semanal:
        break

    best = process.extractOne(
        nb_norm,
        nombres_semanal,
        scorer=fuzz.token_set_ratio
    )

    if best and best[1] >= UMBRAL_FUZZY:
        nm_norm, score = best[0], best[1]
        matches.append((idx, nb_norm, nm_norm, score))

        datos_sem = sem_unmatched_dict.pop(nm_norm)
        nombres_semanal.remove(nm_norm)

        # Copiar columnas del semanal a la fila de la base
        for col in df_sem.columns:
            if col == "__NOMBRE_NORM":
                continue
            if col not in df_join.columns:
                df_join[col] = pd.NA
            df_join.at[idx, col] = datos_sem.get(col, pd.NA)

        df_join.at[idx, "Name"] = datos_sem["Name"]
        df_join.at[idx, "__NOMBRE_NORM"] = nm_norm

print(f"\nCoincidencias fuzzy aplicadas: {len(matches)} (umbral >= {UMBRAL_FUZZY})")
if matches:
    print("Ejemplos:")
    for i, (idx, nb, nm, score) in enumerate(matches[:10], start=1):
        print(f"  {i}. BASE='{nb}'  <->  SEMANAL='{nm}'  (score={score})")

# 6) Proveedores realmente NUEVOS (semanal que no matcheó ni exacto ni fuzzy)
rows_new = []
for nm_norm, datos_sem in sem_unmatched_dict.items():
    new_row = {col: pd.NA for col in df_join.columns}
    new_row["__NOMBRE_NORM"] = nm_norm
    if "Name" in df_join.columns:
        new_row["Name"] = datos_sem.get("Name")
    if "NOMBRE PROVEEDOR" in df_join.columns:
        new_row["NOMBRE PROVEEDOR"] = datos_sem.get("Name")
    for col in df_sem.columns:
        if col in df_join.columns:
            new_row[col] = datos_sem.get(col)
    rows_new.append(new_row)

df_new = pd.DataFrame(rows_new) if rows_new else pd.DataFrame(columns=df_join.columns)

# 7) DataFrame final = base (con matches) + nuevos del semanal
df_final = pd.concat([df_join, df_new], ignore_index=True)

# 8) Ordenar columnas: primero las de la base, luego extras
base_main_cols = [c for c in base_cols_original if c in df_final.columns]
sem_extra_cols = [
    c for c in df_final.columns
    if c not in base_main_cols and c != "__NOMBRE_NORM"
]

df_consolidado = df_final[base_main_cols + sem_extra_cols]
# ----- REGLA DE NEGOCIO: recalcular Nivel de Riesgo según CATEGORIA -----

# Backup del valor original por si quieres comparar luego
df_consolidado["Nivel de Riesgo (HOMOLOGADO CALIFICADORA)_ORIGINAL"] = (
    df_consolidado["Nivel de Riesgo (HOMOLOGADO CALIFICADORA)"]
)

# Aplicar la regla A/B/C/D + REVISAR
df_consolidado["Nivel de Riesgo (HOMOLOGADO CALIFICADORA)"] = (
    df_consolidado["CATEGORIA"].apply(calcular_nivel_riesgo)
)

# 9) Resumen
total_base = len(df_base)
total_sem = len(df_sem)
total_consolidado = len(df_consolidado)

base_norm_set = set(df_base["__NOMBRE_NORM"].dropna().unique())
sem_norm_set = set(df_sem["__NOMBRE_NORM"].dropna().unique())
nuevos_desde_sem = len(sem_norm_set - base_norm_set)
en_ambos = len(base_norm_set & sem_norm_set)

print("\nResumen de consolidación:")
print(f"- Registros en BASE PRINCIPAL: {total_base}")
print(f"- Registros en ARCHIVO SEMANAL: {total_sem}")
print(f"- Proveedores presentes en ambos (por llave normalizada): {en_ambos}")
print(f"- Proveedores que solo están en el SEMANAL (llave normalizada): {nuevos_desde_sem}")
print(f"- Filas totales en archivo consolidado: {total_consolidado}")

# 10) Exportar
nombre_salida = "Proveedores_consolidado_Noviembre21.xlsx"
df_consolidado.to_excel(nombre_salida, index=False)
print(f"\nArchivo generado: {nombre_salida}")

files.download(nombre_salida)


1) Selecciona el archivo BASE PRINCIPAL (Base_Principal_Proveedores.xlsx)


Saving Base_Principal_Proveedores.xlsx to Base_Principal_Proveedores (1).xlsx
-> Archivo cargado: Base_Principal_Proveedores (1).xlsx
Encabezado de la BASE encontrado en la fila 0.

2) Selecciona el archivo SEMANAL (BANCO_GUAYAQUIL_S_A_PROVEEDORES_....xlsx)


Saving BANCO_GUAYAQUIL_S_A_PROVEEDORES_CRITICOS_-_BANCO_GUAYAQUIL_S_A_1763400296 (1).xlsx to BANCO_GUAYAQUIL_S_A_PROVEEDORES_CRITICOS_-_BANCO_GUAYAQUIL_S_A_1763400296 (1) (1).xlsx
-> Archivo cargado: BANCO_GUAYAQUIL_S_A_PROVEEDORES_CRITICOS_-_BANCO_GUAYAQUIL_S_A_1763400296 (1) (1).xlsx
Encabezado del SEMANAL encontrado en la fila 1.

Coincidencias fuzzy aplicadas: 0 (umbral >= 80)

Resumen de consolidación:
- Registros en BASE PRINCIPAL: 50
- Registros en ARCHIVO SEMANAL: 46
- Proveedores presentes en ambos (por llave normalizada): 45
- Proveedores que solo están en el SEMANAL (llave normalizada): 1
- Filas totales en archivo consolidado: 51

Archivo generado: Proveedores_consolidado_Noviembre21.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>